# Lab 16 — Trotter-Suzuki: Intuición y Fórmulas de Alto Orden

Extendemos el lab de trotterización estudiando las **fórmulas de producto de Suzuki** de alto orden.
Veremos cómo las fórmulas simétricas y recursivas mejoran la convergencia.

**Referencia**: Suzuki (1991) — 'General theory of fractal path integrals with applications to many-body theories'

## 1. Familia de fórmulas de Suzuki

Las fórmulas de Suzuki S_k se construyen recursivamente:

- **S₁(t)** = e^{H₁t}e^{H₂t} — error O(t²)
- **S₂(t)** = e^{H₁t/2}e^{H₂t}e^{H₁t/2} — error O(t³)
- **S₄(t)** = S₂(p·t)S₂(p·t)S₂((1-4p)t)S₂(p·t)S₂(p·t) — error O(t⁵)

con p = 1/(4-4^{1/3}).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp

# Constante de Suzuki para S4
p_suzuki = 1 / (4 - 4**(1/3))
print(f'Constante Suzuki p = {p_suzuki:.6f}')
print(f'Coeficiente central: 1 - 4p = {1 - 4*p_suzuki:.6f}')

# Hamiltoniano de prueba: H = XX + ZI + IZ
H = SparsePauliOp.from_list([('XX', 1.0), ('ZI', 0.5), ('IZ', 0.5)])
H1 = SparsePauliOp('XX')
H2 = SparsePauliOp.from_list([('ZI', 0.5), ('IZ', 0.5)])

H_mat = H.to_matrix()
H1_mat = H1.to_matrix()
H2_mat = H2.to_matrix()

print('\nComutador [H1, H2] ≠ 0?')
comm = H1_mat @ H2_mat - H2_mat @ H1_mat
print(f'||[H1,H2]||₂ = {np.linalg.norm(comm):.4f} (≠0 → necesitamos Trotter)')

## 2. Implementación matricial de las fórmulas de Suzuki

Para verificar la convergencia, calculamos el operador de evolución directamente
como producto de matrices exponenciales.

In [ ]:
def exact_evolution(H: np.ndarray, t: float) -> np.ndarray:
    return expm(-1j * H * t)

def trotter_1(H1, H2, t, n):
    dt = t / n
    step = expm(-1j*H1*dt) @ expm(-1j*H2*dt)
    U = np.eye(H1.shape[0], dtype=complex)
    for _ in range(n):
        U = step @ U
    return U

def suzuki_2(H1, H2, t, n):
    dt = t / n
    step = expm(-1j*H1*dt/2) @ expm(-1j*H2*dt) @ expm(-1j*H1*dt/2)
    U = np.eye(H1.shape[0], dtype=complex)
    for _ in range(n):
        U = step @ U
    return U

def suzuki_4(H1, H2, t, n):
    p = p_suzuki
    dt = t / n

    def s2_step(dt_):
        return expm(-1j*H1*dt_/2) @ expm(-1j*H2*dt_) @ expm(-1j*H1*dt_/2)

    step = (s2_step(p*dt) @ s2_step(p*dt) @
            s2_step((1-4*p)*dt) @ s2_step(p*dt) @ s2_step(p*dt))
    U = np.eye(H1.shape[0], dtype=complex)
    for _ in range(n):
        U = step @ U
    return U

# Test rápido
t = 1.0
U_exact = exact_evolution(H_mat, t)
U_t1 = trotter_1(H1_mat, H2_mat, t, n=10)
U_s2 = suzuki_2(H1_mat, H2_mat, t, n=10)
U_s4 = suzuki_4(H1_mat, H2_mat, t, n=10)

def op_error(U1, U2):
    return np.linalg.norm(U1 - U2)

print(f't=1.0, n=10 pasos:')
print(f'Error Trotter 1°: {op_error(U_exact, U_t1):.2e}')
print(f'Error Suzuki 2°:  {op_error(U_exact, U_s2):.2e}')
print(f'Error Suzuki 4°:  {op_error(U_exact, U_s4):.2e}')

## 3. Convergencia: error de operador vs número de pasos

Graficamos el error ||U_approx - U_exact||₂ en escala log-log.
La pendiente confirma el orden de convergencia de cada fórmula.

In [ ]:
t_fixed = 2.0
n_values = [1, 2, 4, 8, 16, 32]
U_ref = exact_evolution(H_mat, t_fixed)

errors = {'Trotter 1°': [], 'Suzuki 2°': [], 'Suzuki 4°': []}
for n in n_values:
    errors['Trotter 1°'].append(op_error(U_ref, trotter_1(H1_mat, H2_mat, t_fixed, n)))
    errors['Suzuki 2°'].append(op_error(U_ref, suzuki_2(H1_mat, H2_mat, t_fixed, n)))
    errors['Suzuki 4°'].append(op_error(U_ref, suzuki_4(H1_mat, H2_mat, t_fixed, n)))

fig, ax = plt.subplots(figsize=(8, 5))
styles = {'Trotter 1°': 'bo-', 'Suzuki 2°': 'rs-', 'Suzuki 4°': 'g^-'}
for name, errs in errors.items():
    ax.loglog(n_values, errs, styles[name], label=name, markersize=8, linewidth=2)

# Líneas de referencia
ns = np.array(n_values)
ax.loglog(ns, 0.5*(1/ns)**2, 'b--', alpha=0.4, label='O(1/n²)')
ax.loglog(ns, 0.1*(1/ns)**4, 'r--', alpha=0.4, label='O(1/n⁴)')
ax.loglog(ns, 0.01*(1/ns)**8, 'g--', alpha=0.4, label='O(1/n⁸)')

ax.set_xlabel('Número de pasos n', fontsize=12)
ax.set_ylabel('Error ||U_approx - U_exact||', fontsize=12)
ax.set_title(f'Convergencia fórmulas de Suzuki (t={t_fixed})', fontsize=13)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Coste en puertas vs precisión

Las fórmulas de mayor orden necesitan más exponenciales por paso:
- S₁: 2 exponenciales/paso
- S₂: 3 exponenciales/paso
- S₄: 5·3 = 15 exponenciales/paso

La pregunta real es: ¿qué fórmula da más precisión por puerta?

In [ ]:
print('Análisis coste-precisión (t=2, error objetivo ≤ 1e-4):')
print(f'{"Fórmula":<12} | Pasos n | Exp/paso | Total exp | Error')
print('-' * 55)

target_error = 1e-4
costs = {'Trotter 1°': 2, 'Suzuki 2°': 3, 'Suzuki 4°': 15}

for name, exp_per_step in costs.items():
    errs = errors[name]
    # Encontrar el n mínimo que alcanza el objetivo
    n_needed = None
    for i, n in enumerate(n_values):
        if errs[i] <= target_error:
            n_needed = n
            final_err = errs[i]
            break
    if n_needed:
        total = n_needed * exp_per_step
        print(f'{name:<12} | {n_needed:<7} | {exp_per_step:<8} | {total:<9} | {final_err:.2e}')
    else:
        print(f'{name:<12} | >32    | {exp_per_step:<8} | >?       | no alcanzado')

## 5. Resumen

| Fórmula | Error/paso | Exp/paso | Ideal para |
|---------|-----------|---------|------------|
| Trotter S₁ | O(δt²) | 2m | Prototipado rápido |
| Suzuki S₂ | O(δt³) | 2m-1 | Balance coste-precisión |
| Suzuki S₄ | O(δt⁵) | 5(2m-1) | Alta precisión, t grande |

**Regla para hardware**: en procesadores NISQ, el error de puerta domina antes de que
el error Trotter importe. Usar S₂ es casi siempre suficiente y más barato que S₄.

In [ ]:
# Verificación numérica de los órdenes de convergencia
print('Verificación de órdenes de convergencia:')
for name, errs in errors.items():
    # Pendiente log-log entre n=2 y n=32
    log_n = np.log(n_values[1:])
    log_e = np.log([max(e, 1e-15) for e in errs[1:]])
    slope = np.polyfit(log_n, log_e, 1)[0]
    print(f'  {name}: pendiente = {slope:.2f} (esperada: Trotter=-2, S2=-4, S4=-8... sin considerar constantes)')